# Gradient Inspection, 2020 TensorFlow 2.2 Style

This notebook is a cleaned-up version of an old 2020 note. The point is historical fidelity, not modern style: TensorFlow 2.x existed, but older TensorFlow 1 graph habits were still common when inspecting internals.

The source note is preserved in `original-2020-gradients.md`. This notebook keeps the important ideas while fixing a few broken variable names and making the examples self-contained.

## Why This Notebook Uses `compat.v1`

I was wrestling with the TF1-to-TF2 transition:

- I originally learned TensorFlow 1.x (circa late 2016, early 2017), which used static graphs.
- I often used Keras (prior to `tf.keras`) to make things easier because -- let's face it -- TensorFlow 1.x was a pain to use.
- After having spent a few months or so wrestling with TensorFlow/Keras, 
  I started hearing about how great PyTorch was, but stubbornly continued to use TensorFlow/Keras because 
  I had invested so much time in it.
- Imagine my relief when TensorFlow 2.x came out and made things much more PyTorch-like, with eager execution and a more 
  Pythonic feel. 
- I was glad that I did not have to learn PyTorch, that Keras went all in on TensorFlow (`tf.keras`), and
  that TensorFlow 2.x was easier to use in general when wanting to do more than Keras-style high-level modeling.
- The transition between TensorFlow 1.x and 2.x was not always smooth, however, because of the following reasons:
    - Sometimes TensorFlow 2 code ran eagerly.
    - Sometimes older Keras backend examples expected a static graph.
    - Sometimes `tf.compat.v1.disable_eager_execution()` was the practical bridge.

I stuck with TensorFlow 2.x for a while, but kept getting frustrated that nearly 
every deep learning paper I read (perhaps every paper!) shared their code in PyTorch, leaving me
trying to translate it to TensorFlow before getting a chance to tinker with it and determine if it
was worth my time. When I finally did give in and learn PyTorch, my impression was that 
TensorFlow 2.x literally tried its best to be PyTorch.

This notebook was originally created during the TF1-to-TF2 transition, which is why this notebook 
intentionally disables eager execution. Run it in the `legacy2020` environment, not the modern environment.

In [ ]:
import numpy as np
import tensorflow.compat.v1 as tf

# This is the historically important line. Once eager execution is disabled in
# a Python process, you should restart the kernel to go back to normal TF2 eager mode.
tf.disable_eager_execution()

K = tf.keras.backend
np.random.seed(7)
tf.set_random_seed(7)

print('TensorFlow:', tf.__version__)
print('Eager execution:', tf.executing_eagerly())

## Tiny Self-Contained Data

The old note used MNIST/CIFAR helpers. For preservation, this notebook uses synthetic 28x28 images so it can run without downloading datasets.

In [ ]:
def make_toy_images(n=64, image_shape=(28, 28), num_classes=10):
    rng = np.random.RandomState(7)
    x = rng.normal(loc=0.0, scale=0.2, size=(n, *image_shape)).astype('float32')
    y = (np.arange(n) % num_classes).astype('int32')

    # Give each class a small bright square at a class-dependent location. This
    # is not a serious dataset; it just creates a stable signal for gradient demos.
    for i, label in enumerate(y):
        row = 2 + (label % 5) * 4
        col = 2 + (label // 5) * 10
        x[i, row:row + 4, col:col + 4] += 1.0

    return x, y

x_data, y_data = make_toy_images()
x_gradtest1 = x_data[:1]
y_gradtest1 = y_data[:1]
x_gradtest2 = x_data[:10]
y_gradtest2 = y_data[:10]

print(x_data.shape, y_data.shape)
print('Example labels:', y_data[:10])

## Model Helper

This keeps the spirit of the original helper: a small Keras model with trainable convolution, normalization, and dense layers.

In [ ]:
def get_model(channel_depth=1):
    inputs = tf.keras.layers.Input((28, 28), name='image')
    x = tf.keras.layers.Reshape((28, 28, channel_depth), name='add_channel')(inputs)
    x = tf.keras.layers.BatchNormalization(name='batch_norm_0')(x)
    x = tf.keras.layers.Conv2D(
        filters=8,
        kernel_size=3,
        padding='same',
        use_bias=False,
        name='conv_1',
    )(x)
    x = tf.keras.layers.BatchNormalization(name='batch_norm_1')(x)
    x = tf.keras.layers.ReLU(name='relu_1')(x)
    x = tf.keras.layers.MaxPool2D(pool_size=2, strides=2, padding='same', name='pool_1')(x)
    x = tf.keras.layers.Flatten(name='flatten')(x)
    x = tf.keras.layers.Dense(16, name='dense_1')(x)
    x = tf.keras.layers.ReLU(name='relu_2')(x)
    outputs = tf.keras.layers.Dense(10, activation='softmax', name='class_probs')(x)

    model = tf.keras.models.Model(inputs, outputs, name='tf22_style_probe_model')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )
    return model

K.clear_session()
model = get_model()
model.summary()

# In graph mode, make sure variables exist before backend functions evaluate them.
K.get_session().run(tf.global_variables_initializer())

## Activations And Gradients With Respect To A Model Score

The old note called this a kind of camera inside the network. That is a good mental model: fetch intermediate layer outputs on the forward pass, then fetch how a selected score changes with respect to those outputs on the backward pass.

In [ ]:
trainable_layers = [layer for layer in model.layers if layer.trainable_weights]
layer_outputs = [layer.output for layer in trainable_layers]

# Pick one scalar score. Keras backend gradients are easiest to reason about
# when the target is scalar.
class_zero_score = K.sum(model.output[:, 0])
grad_tensors = K.gradients(class_zero_score, layer_outputs)

layer_probe = K.function(
    [model.input, K.learning_phase()],
    layer_outputs + grad_tensors,
)

values = layer_probe([x_gradtest1, 0])
activation_values = values[:len(layer_outputs)]
activation_grads = values[len(layer_outputs):]

for layer, activation, grad in zip(trainable_layers, activation_values, activation_grads):
    print(f'{layer.name:14} activation={activation.shape!s:18} gradient={grad.shape!s}')

## Trainable Weight Gradients With Respect To Loss

This is the usual training question: how should each trainable weight move to reduce the loss?

In [ ]:
labels = K.placeholder(shape=(None,), dtype='int32', name='labels')
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
loss_value = K.mean(loss_fn(labels, model.output))

trainable_weights = model.trainable_weights
weight_grad_tensors = K.gradients(loss_value, trainable_weights)
weight_probe = K.function(
    [model.input, labels, K.learning_phase()],
    weight_grad_tensors,
)

weight_grads = weight_probe([x_gradtest2, y_gradtest2, 0])

for weight, grad in zip(trainable_weights, weight_grads):
    print(f'{weight.name:34} weight={weight.shape!s:16} grad={grad.shape!s}')

## Histogram Summaries

The original note printed large tensors directly. That is useful once, but summaries are usually better for repeated inspection.

In [ ]:
def summarize_array(name, value):
    value = np.asarray(value)
    return {
        'name': name,
        'shape': value.shape,
        'min': float(np.min(value)),
        'mean': float(np.mean(value)),
        'max': float(np.max(value)),
        'std': float(np.std(value)),
    }

summaries = []
for layer, activation, grad in zip(trainable_layers, activation_values, activation_grads):
    summaries.append(summarize_array(f'{layer.name}.activation', activation))
    summaries.append(summarize_array(f'{layer.name}.gradient', grad))

for item in summaries:
    print(item)

## Modernized Companion Notebook (2026)

The modernized companion notebook repeats the same loop with TensorFlow `GradientTape` and PyTorch `autograd`.